# Olist: monthly order count

## Goal
Define "monthly orders" precisely and compute it. The SQL version is in `sql/01_monthly_orders.sql`. Table grain and keys are covered in [01-data-start](01-data-start.ipynb).

## Setup
Only the orders table is needed.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

# Project folder = the nearest parent that contains data/raw
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw").is_dir())
DATA_DIR = ROOT / "data" / "raw"

orders = pd.read_csv(DATA_DIR / "olist_orders_dataset.csv")
print("orders:", orders.shape)

orders: (99441, 8)


## Definitions
- **Time**: `order_purchase_timestamp`, parsed into `purchase_ts`, then truncated to month.
- **Status**: `canceled` and `unavailable` are excluded; every other status is kept.
- **Unit**: one row per order, counted as distinct `order_id`.
- **Checks**: no timestamp fails to parse; valid orders are 98,207 (99,441 − 1,234) and the months add up to the same number.

In [2]:
# Status distribution, used to define "valid orders"
display(orders["order_status"].value_counts(dropna=False).rename_axis("order_status").reset_index(name="n_orders"))

,order_status,n_orders
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2


In [3]:
# Parse the purchase time; failures become NaT instead of being dropped silently
orders["purchase_ts"] = pd.to_datetime(orders["order_purchase_timestamp"], errors="coerce")

# Missing values before and after parsing: both should be 0
print("Missing order_purchase_timestamp before parsing:", orders["order_purchase_timestamp"].isna().sum())
print("Missing purchase_ts after parsing:", orders["purchase_ts"].isna().sum())

Missing order_purchase_timestamp before parsing: 0
Missing purchase_ts after parsing: 0


In [4]:
# Exclude canceled and unavailable orders
valid_orders = orders[~orders["order_status"].isin(["canceled", "unavailable"])]

print("All orders:", len(orders))
print("Valid orders:", len(valid_orders), f"({len(orders) - len(valid_orders)} excluded)")

All orders: 99441
Valid orders: 98207 (1234 excluded)


In [5]:
# Monthly order count: distinct order_id per purchase month
valid_orders = valid_orders.copy()
valid_orders["purchase_month"] = valid_orders["purchase_ts"].dt.to_period("M")

monthly_orders = valid_orders.groupby("purchase_month")["order_id"].nunique()

# Print every month to spot gaps and check the first and last months
print(monthly_orders.to_string())

# Check: the months must add up to the number of valid orders
print("Sum of months:", monthly_orders.sum(), "| valid orders:", len(valid_orders), "| equal:", monthly_orders.sum() == len(valid_orders))

purchase_month
2016-09       2
2016-10     293
2016-12       1
2017-01     787
2017-02    1718
2017-03    2617
2017-04    2377
2017-05    3640
2017-06    3205
2017-07    3946
2017-08    4272
2017-09    4227
2017-10    4547
2017-11    7423
2017-12    5620
2018-01    7187
2018-02    6625
2018-03    7168
2018-04    6919
2018-05    6833
2018-06    6145
2018-07    6233
2018-08    6421
2018-09       1
Freq: M
Sum of months: 98207 | valid orders: 98207 | equal: True


## Findings
- 2016-11 has no orders at all. 2016-09 (2 orders), 2016-12 (1) and 2018-09 (1) are edge months and are left out of any trend.
- Observation period: 2017-01 to 2018-08, where every month has orders.
- The peak is 2017-11 with 7,423 orders, falling back to 5,620 in 2017-12. The data does not give the reason for the peak, so none is claimed.
- From 2018-01 to 2018-08 the count stays between roughly 6,100 and 7,200 orders a month.